# Generate environment

By joining different sources of data:
- Weather data for TMY
- Electricity price data
- Thermal load data

And save them as a unified dataset.

In [2]:
from typing import Literal
from pathlib import Path
import pandas as pd
import datetime
from loguru import logger

%load_ext autoreload
%autoreload 2

# Paths
data_path: Path = Path("../../data/datasets/partial")
data_output_path: Path = Path("../../data/datasets")
viz_output_path: Path = Path("../results/visualizations")
# Parameters
environment_id: Literal["psa", "andasol", "med_eds", "psa_andasol"] = "psa_andasol"

params_dict = {
    "psa": {
        "weather": "tmy_2005_tabernas.h5",
        "electricity": "electricity_price_data_20211231_20241231.h5",
        "load": "thermal_load_data_psa_20220101_20241231.h5",
        "water": "water_data_psa_20220101_20241231.h5",
        "year_start": 2022,
        "year_end": 2024,
        "constraint_Tv_to_Tamb": True,
    },
    "psa_andasol": {
        "weather": "tmy_2005_tabernas.h5",
        "electricity": "electricity_price_data_20211231_20241231.h5",
        "load": "thermal_load_data_andasol_downscaled_20220101_20241231.h5",
        "water": "water_data_psa_20220101_20241231.h5",
        "year_start": 2022,
        "year_end": 2024,
        "constraint_Tv_to_Tamb": False,
    },
    "andasol": {
        "weather": "tmy_2005_guadix.h5",
        "electricity": "electricity_price_data_20211231_20241231.h5",
        "load": "thermal_load_data_andasol_20220101_20241231.h5",
        "water": "water_data_andasol_20220101_20241231.h5",
        "year_start": 2022,
        "year_end": 2024,
        "constraint_Tv_to_Tamb": False,
    },
    "med_eds": {
        "weather": "tmy_2005_Borg_El_Arab.h5",
        "electricity": "electricity_price_data_20211231_20241231.h5",
        "load": "thermal_load_data_med_eds_20220101_20241231.h5",
        "water": "water_data_med_eds_20220101_20241231.h5",
        "year_start": 2022,
        "year_end": 2024,
        "constraint_Tv_to_Tamb": True,
    },
}

assert environment_id in params_dict, f"Environment ID '{environment_id}' not recognized. Available: {list(params_dict.keys())}"

params = params_dict[environment_id]
year_start: int = params["year_start"]
year_end: int = params["year_end"]
constraint_Tv_to_Tamb: bool = params.get("constraint_Tv_to_Tamb", False)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Load datasets
df_weather = pd.read_hdf(data_path / params["weather"]) if params["weather"].endswith("h5") else pd.read_csv(data_path / params["weather"], index_col=0, parse_dates=True)
df_electricity = pd.read_hdf(data_path / params["electricity"]) if params["electricity"].endswith("h5") else pd.read_csv(data_path / params["electricity"], index_col=0, parse_dates=True)
df_load = pd.read_hdf(data_path / params["load"]) if params["load"].endswith("h5") else pd.read_csv(data_path / params["load"], index_col=0, parse_dates=True)
df_water = pd.read_hdf(data_path / params["water"]) if params["water"].endswith("h5") else pd.read_csv(data_path / params["water"], index_col=0, parse_dates=True)

# Create a pandas series with one-hour resolution
start_date = datetime.datetime(year_start, 1, 1, 0, 0, 0, tzinfo=datetime.timezone.utc)
end_date = datetime.datetime(year_end, 12, 31, 23, 0, 0, tzinfo=datetime.timezone.utc)
date_rng = pd.date_range(start=start_date, end=end_date, freq='h', tz='UTC', inclusive="both")

display(df_weather.head())
display(df_electricity.head())
display(df_load.head())
display(df_water.head())


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td
time,,,,,,,,,,
2005-01-01 00:00:00+00:00,0,0,0,0,10.7,90,0.0,-144.0,6.7,9.1
2005-01-01 01:00:00+00:00,0,0,0,0,11.6,82,0.0,-163.4,0.5,8.7
2005-01-01 02:00:00+00:00,0,0,0,0,11.3,82,0.0,-124.4,0.5,8.4
2005-01-01 03:00:00+00:00,0,0,0,0,11.2,82,0.0,-105.4,0.3,8.3
2005-01-01 04:00:00+00:00,0,0,0,0,11.0,82,0.0,-93.6,0.2,8.1


,Ce_pvpc_eur_MWh,Ce_spot_market_price_eur_MWh,Ce_pvpc_eur_kWh,Ce_spot_market_price_eur_kWh
datetime,,,,
2021-12-31 23:00:00+00:00,204.51,145.86,0.20451,0.14586
2022-01-01 00:00:00+00:00,171.35,114.90,0.17135,0.11490
2022-01-01 01:00:00+00:00,172.70,113.87,0.17270,0.11387
2022-01-01 02:00:00+00:00,156.07,97.80,0.15607,0.09780
2022-01-01 03:00:00+00:00,159.08,97.80,0.15908,0.09780


,Q_kW,Tv_C,mv_kgh
2022-01-01 00:00:00+00:00,0.0,0.0,0.0
2022-01-01 01:00:00+00:00,0.0,0.0,0.0
2022-01-01 02:00:00+00:00,0.0,0.0,0.0
2022-01-01 03:00:00+00:00,0.0,0.0,0.0
2022-01-01 04:00:00+00:00,0.0,0.0,0.0


,water_price_eur_m3,water_price_alternative_eur_m3,water_price_eur_l,water_price_alternative_eur_l,Vavail_m3,Vavail_l
2022-01-01 00:00:00+00:00,0.029054,9.1920,0.000029,0.009192,0.886583,886.583184
2022-01-01 01:00:00+00:00,0.029054,9.1096,0.000029,0.009110,0.886583,886.583184
2022-01-01 02:00:00+00:00,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
2022-01-01 03:00:00+00:00,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
2022-01-01 04:00:00+00:00,0.029054,7.6592,0.000029,0.007659,0.886583,886.583184


In [4]:
# Electricity price and thermal load data should be available for the selected period
# Raise an error if the data is not available
if df_electricity.index[-1] < end_date:
    logger.warning(f"Electricity price data is not long enough: {df_electricity.index[-1]} < {end_date}. Filling difference")
    df_electricity = pd.concat([df_electricity, pd.Series(index=pd.date_range(start=df_electricity.index[-1] + datetime.timedelta(hours=1), end=end_date, freq='h', tz='UTC'))])
    df_electricity = df_electricity.ffill()
    
assert df_electricity.index[0] <= start_date and df_electricity.index[-1] >= end_date, f"Electricity price data is not available for the selected period: {df_electricity.index[0]} - {df_electricity.index[-1]}"
assert df_load.index[0] <= start_date and df_load.index[-1] >= end_date, f"Thermal load data is not available for the selected period: {df_load.index[0]} - {df_load.index[-1]}"

# Weather data needs to be adapted to the selected period (modify its year)
# Repeat data for each year
# if df_weather.index[0] <= start_date and df_weather.index[-1] >= end_date:
df_weather_ = df_weather.copy()
df_weather_ = pd.concat([df_weather_] * (year_end - year_start + 1))
df_weather_ = df_weather_.set_index(date_rng[:len(df_weather_)]).reindex(date_rng).ffill()

# Check if the weather data is available for the selected period
assert df_weather_.index[0] <= start_date and df_weather_.index[-1] >= end_date, f"Weather data is not available for the selected period: {df_weather.index[0]} - {df_weather.index[-1]}"
df_weather = df_weather_

if constraint_Tv_to_Tamb:
    # Modify Tv so that Tv always satisfies the condition Tv < Tamb - 5
    violations = df_load[df_load['Tv_C'] > df_weather['Tamb_C']+ 5 ]

    # Check if any violations exist
    if not violations.empty:
        logger.warning(f"Found {len(violations)} violations of Tv+5<Tamb")

    df_load['Tv_C'] = df_load['Tv_C'].where(df_load['Tv_C'] >= df_weather['Tamb_C'] + 5, df_weather['Tamb_C'] + 5)



2025-09-15 09:47:36.766 | WARNING  | __main__:<module>:4 - Electricity price data is not long enough: 2024-12-31 22:00:00+00:00 < 2024-12-31 23:00:00+00:00. Filling difference


In [5]:
# Join datasets
df = pd.concat([df_weather, df_electricity, df_load, df_water], axis=1, join="inner")

logger.info(f"Resulting dataframe has {len(df)} rows")

# Save to hdf5 and csv
fname = f"environment_data_{environment_id}_{df.index[0].strftime('%Y%m%d')}_{df.index[-1].strftime('%Y%m%d')}"
df.sort_index(inplace=True)
df.to_hdf(data_output_path / f"{fname}.h5", key="data", mode="w", format="table")
df.to_csv(data_output_path / f"{fname}.csv")
    
logger.info(f"Saved {fname} to {data_output_path}")
df


2025-09-15 09:47:39.652 | INFO     | __main__:<module>:4 - Resulting dataframe has 26304 rows


2025-09-15 09:47:40.208 | INFO     | __main__:<module>:12 - Saved environment_data_psa_andasol_20220101_20241231 to ../../data/datasets


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,0,Q_kW,Tv_C,mv_kgh,water_price_eur_m3,water_price_alternative_eur_m3,water_price_eur_l,water_price_alternative_eur_l,Vavail_m3,Vavail_l
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,10.7,90.0,0.0,-144.0,6.7,9.1,...,NaN,0.0,0.0,0.0,0.029054,9.1920,0.000029,0.009192,0.886583,886.583184
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,11.6,82.0,0.0,-163.4,0.5,8.7,...,NaN,0.0,0.0,0.0,0.029054,9.1096,0.000029,0.009110,0.886583,886.583184
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,11.3,82.0,0.0,-124.4,0.5,8.4,...,NaN,0.0,0.0,0.0,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,11.2,82.0,0.0,-105.4,0.3,8.3,...,NaN,0.0,0.0,0.0,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,11.0,82.0,0.0,-93.6,0.2,8.1,...,NaN,0.0,0.0,0.0,0.029054,7.6592,0.000029,0.007659,0.886583,886.583184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,12.4880,0.000029,0.012488,0.991771,991.771020
2024-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,12.0680,0.000029,0.012068,0.991771,991.771020
2024-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,11.5520,0.000029,0.011552,0.991771,991.771020
2024-12-31 22:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,11.1496,0.000029,0.011150,0.991771,991.771020


In [6]:
df_env = pd.read_hdf(data_output_path / f"{fname}.h5")
df_env


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,0,Q_kW,Tv_C,mv_kgh,water_price_eur_m3,water_price_alternative_eur_m3,water_price_eur_l,water_price_alternative_eur_l,Vavail_m3,Vavail_l
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,10.7,90.0,0.0,-144.0,6.7,9.1,...,NaN,0.0,0.0,0.0,0.029054,9.1920,0.000029,0.009192,0.886583,886.583184
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,11.6,82.0,0.0,-163.4,0.5,8.7,...,NaN,0.0,0.0,0.0,0.029054,9.1096,0.000029,0.009110,0.886583,886.583184
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,11.3,82.0,0.0,-124.4,0.5,8.4,...,NaN,0.0,0.0,0.0,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,11.2,82.0,0.0,-105.4,0.3,8.3,...,NaN,0.0,0.0,0.0,0.029054,7.8240,0.000029,0.007824,0.886583,886.583184
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,11.0,82.0,0.0,-93.6,0.2,8.1,...,NaN,0.0,0.0,0.0,0.029054,7.6592,0.000029,0.007659,0.886583,886.583184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,12.4880,0.000029,0.012488,0.991771,991.771020
2024-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,12.0680,0.000029,0.012068,0.991771,991.771020
2024-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,11.5520,0.000029,0.011552,0.991771,991.771020
2024-12-31 22:00:00+00:00,0.0,0.0,0.0,0.0,6.0,98.0,0.0,115.2,1.6,5.7,...,NaN,0.0,0.0,0.0,0.029054,11.1496,0.000029,0.011150,0.991771,991.771020


In [7]:
from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

var_ids: list[str] = [
    ["Tamb_C", "Tv_C"], 
    "HR_pct", "precip_mm", "Q_kW", 
    ["Ce_pvpc_eur_MWh", "Ce_spot_market_price_eur_MWh"],
    ["water_price_eur_m3", "water_price_alternative_eur_m3"],
    "Vavail_m3",
]
var_units: list[str] = ["°C", "%", "mm", "kW<sub>th</sub>", "€/MW<sub>e</sub>", "€/m<sup>3</sup>","m<sup>3</sup>"]

fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
fig = FigureWidgetResampler(fig)

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    var_id = [var_id] if not isinstance(var_id, list) else var_id
    
    for v_id in var_id:
        fig.add_trace(
            go.Scattergl(name=v_id.replace("_", " "), showlegend=True), 
            hf_x=df_env.index,
            hf_y=np.ascontiguousarray( df_env[v_id] ), 
            # max_n_samples=2_000,
            row=i + 1, col=1
        )
    fig.update_yaxes(title_text=var_unit, row=i + 1)

fig.update_layout(
    height=1400,
    width=800,
    title="<b>Environment variables</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.15, xanchor="left", x=0),
    margin=dict(l=20, r=20, t=340, b=20),
)

fig


FigureWidgetResampler({
    'data': [{'name': '<b style="color:sandybrown">[R]</b> Tamb C <i style="color:#fc9944">~1D</i>',
              'showlegend': True,
              'type': 'scattergl',
              'uid': 'cc7d2e8b-82f9-4900-b6cc-34743b54e343',
              'x': array([datetime.datetime(2022, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 1, 13, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 3, 5, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2024, 12, 29, 15, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 30, 23, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 31, 23, 0, tzinfo=datetime.timezone.utc)],
                         shape=(1000,), dtype=object),
              'xaxis': 'x',
              'y': array([10.7, 13.4,  5.6, ..., 15.3,  6. ,  6. ], shape=(1000,)),
    

In [49]:
from phd_visualizations import save_figure

if fig.layout.xaxis.range is not None:
    start, end = fig.layout.xaxis.range
else:
    start, end = df.index[0].strftime("%Y-%m-%d %H:%M:%S.%f")[:-2], df.index[-1].strftime("%Y-%m-%d %H:%M:%S.%f")[:-2]

save_figure(fig, f"solhycool_env_viz_{environment_id}_{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", 
            figure_path=viz_output_path, formats=["png", "svg"])


2025-04-15 09:55:28.202 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/visualizations')]/solhycool_env_viz_psa_20220101_20241231.png
2025-04-15 09:55:33.855 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/visualizations')]/solhycool_env_viz_psa_20220101_20241231.svg
